# House Price Prediction Using Machine Learning

B.Tech Summer Training Project (Amazon ML Summer School 2026)

This notebook walks through the same pipeline as `src/train_model.py`, step by step, for exploration and for generating report screenshots.

**Before running:** place `Delhi house data.csv` inside the `data/` folder (see `data/README.md`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('../data/Delhi house data.csv')
df = pd.read_csv(DATA_PATH)
df.head()


## 1. Dataset Inspection

In [ ]:
print('Shape:', df.shape)
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.duplicated().sum()


In [ ]:
df.describe(include='all')


## 2. Data Cleaning

Keep only the columns this project needs, coerce numeric types, drop duplicates/invalid rows.

In [ ]:
work = df[['Area', 'BHK', 'Bathroom', 'Locality', 'Price']].copy()

for c in ['Area', 'BHK', 'Bathroom', 'Price']:
    work[c] = pd.to_numeric(work[c].astype(str).str.replace(',', '', regex=False), errors='coerce')

work['Locality'] = work['Locality'].astype(str).str.strip()

before = len(work)
work = work.drop_duplicates()
work = work.dropna(subset=['Price', 'Area', 'BHK', 'Bathroom', 'Locality'])
work = work[(work['Area'] > 0) & (work['BHK'] > 0) & (work['Bathroom'] > 0) & (work['Price'] > 0)]
work = work.reset_index(drop=True)

print(f'Rows before cleaning: {before}')
print(f'Rows after cleaning : {len(work)}')


## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(work['Price'], bins=40)
plt.title('Distribution of House Prices')
plt.xlabel('Price'); plt.ylabel('Count')
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(work['Area'], work['Price'], alpha=0.4, s=15)
plt.title('Area vs Price')
plt.xlabel('Area (sqft)'); plt.ylabel('Price')
plt.show()


In [ ]:
work.boxplot(column='Price', by='BHK', figsize=(7,5))
plt.title('BHK vs Price'); plt.suptitle('')
plt.show()


In [ ]:
work.boxplot(column='Price', by='Bathroom', figsize=(7,5))
plt.title('Bathrooms vs Price'); plt.suptitle('')
plt.show()


## 4. Feature Preparation, Train/Test Split, and Model Training

Uses a scikit-learn `ColumnTransformer` + `Pipeline` so the exact same preprocessing is reused by `app.py`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

X = work[['Area', 'BHK', 'Bathroom', 'Locality']]
y = work['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['Area', 'BHK', 'Bathroom']
categorical_features = ['Locality']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

pipeline = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
pipeline.fit(X_train, y_train)


## 5. Model Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'Training samples: {len(X_train)}')
print(f'Testing samples : {len(X_test)}')
print(f'R2 Score : {r2:.4f}')
print(f'MAE      : {mae:,.2f}')
print(f'RMSE     : {rmse:,.2f}')


## 6. Actual vs Predicted and Residual Plots

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_test, y_pred, alpha=0.4, s=15)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, color='red')
plt.title('Actual vs Predicted Price')
plt.xlabel('Actual Price'); plt.ylabel('Predicted Price')
plt.show()


In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(7,5))
plt.scatter(y_pred, residuals, alpha=0.4, s=15)
plt.axhline(0, color='red')
plt.title('Residual Plot')
plt.xlabel('Predicted Price'); plt.ylabel('Residual')
plt.show()


## 7. Save the Trained Pipeline

This notebook is for exploration. To regenerate `model/house_price_model.pkl`, `model/localities.json`, `model/results.json`, and all plots in `images/` in one go, run from the project root:

```
python src/train_model.py
```